# Building and Evaluating a Personal Inflation Index

This notebook walks through the core workflow of this project:

1. Define a household spending basket (category weights).
2. Load (or simulate) category price data over time.
3. Compute a personal Laspeyres price index.
4. Compare it against an illustrative "official CPI"-style reference series.

**Disclosure:** every weight and price series used in this notebook is
either a clearly-labeled illustrative example or deterministic synthetic
data (seed-based), never real MoSPI CPI data. See
`docs/DATA_SOURCES.md` for where to find the real series, and
`docs/METHODOLOGY.md` for the full economic reasoning behind this
project's research question.


In [ ]:
import sys
from pathlib import Path

# Make the local `personal_inflation` package importable without installing it
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from personal_inflation.index import (
    DEFAULT_CATEGORIES,
    laspeyres_index,
    personal_index_time_series,
    compare_to_reference,
)
from personal_inflation.example_data import EXAMPLE_ILLUSTRATIVE_WEIGHTS

DEFAULT_CATEGORIES


## 1. A hand-computed sanity check

Before trusting the library function, let's hand-verify the Laspeyres
formula on a tiny 3-category basket.

Basket: food (w=0.5), housing (w=0.3), transport (w=0.2).

| category | base price | current price | price relative |
|---|---|---|---|
| food | 100 | 110 | 1.10 |
| housing | 200 | 220 | 1.10 |
| transport | 50 | 40 | 0.80 |

By hand: `I = 100 * (0.5*1.10 + 0.3*1.10 + 0.2*0.80) = 100 * 1.04 = 104.0`


In [ ]:
weights = {"food": 0.5, "housing": 0.3, "transport": 0.2}
base_prices = {"food": 100.0, "housing": 200.0, "transport": 50.0}
current_prices = {"food": 110.0, "housing": 220.0, "transport": 40.0}

result = laspeyres_index(base_prices, current_prices, weights)
print(f"Computed index: {result}")
assert abs(result - 104.0) < 1e-9
print("Matches the hand-computed value of 104.0")


## 2. An illustrative example basket

`EXAMPLE_ILLUSTRATIVE_WEIGHTS` is a set of round, hand-picked placeholder
weights shipped with this project purely to demonstrate the methodology.
**These are not official MoSPI weights.** In real use, you would replace
this dict with your own household's actual expenditure shares.


In [ ]:
EXAMPLE_ILLUSTRATIVE_WEIGHTS


## 3. Loading the synthetic category price panel

`data/synthetic_category_price_index.csv` is generated by
`examples/generate_synthetic_data.py` from a fixed random seed. It is
**not real price data** -- it exists only so this notebook has a
multi-period panel to compute a personal index *over time* against.


In [ ]:
data_path = Path.cwd().parent / "data" / "synthetic_category_price_index.csv"
panel = pd.read_csv(data_path, index_col="period")
print(panel.shape)
panel.head()


In [ ]:
reference_series = panel["official_cpi_reference"]
price_df = panel.drop(columns=["official_cpi_reference"])
price_df.head()


## 4. Computing the personal index over time

We use the illustrative example weights to compute a personal index
across every period in the synthetic panel, with the first period as
the base (so it reads exactly 100.0).


In [ ]:
personal_series = personal_index_time_series(price_df, EXAMPLE_ILLUSTRATIVE_WEIGHTS)
print(f"Base period value: {personal_series.iloc[0]:.4f} (should be exactly 100)")
personal_series.tail()


## 5. Comparing against the illustrative reference series

`compare_to_reference` lines up the personal index against the
synthetic `official_cpi_reference` column and reports the gap in both
index points and percentage terms.


In [ ]:
comparison = compare_to_reference(personal_series, reference_series)
comparison.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(comparison.index, comparison["personal_index"], label="Personal index (example weights)", color="#2a78d6", linewidth=2)
ax.plot(comparison.index, comparison["reference_index"], label="Illustrative comparison index (synthetic)", color="#eb6834", linewidth=2, linestyle="--")
ax.set_xticks(ax.get_xticks()[::6])
ax.set_ylabel("Index (base = 100)")
ax.set_title("Personal vs. illustrative comparison index (SYNTHETIC DEMO DATA)")
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Demonstrating heterogeneity: a food-heavy vs. education-heavy basket

To illustrate the actual research question -- why personal inflation can
diverge from an "average" index -- let's compare two very different
hypothetical households: one that spends heavily on food, and one that
spends heavily on education (which, in our synthetic data, drifts up in
price noticeably faster than food on average).


In [ ]:
food_heavy_weights = {"food": 0.70, "housing": 0.15, "other": 0.15}
education_heavy_weights = {"education": 0.50, "housing": 0.30, "other": 0.20}

food_heavy_series = personal_index_time_series(price_df, food_heavy_weights)
education_heavy_series = personal_index_time_series(price_df, education_heavy_weights)

print(f"Food-heavy household, final period index:      {food_heavy_series.iloc[-1]:.2f}")
print(f"Education-heavy household, final period index: {education_heavy_series.iloc[-1]:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(food_heavy_series.index, food_heavy_series.values, label="Food-heavy household", color="#2a78d6", linewidth=2)
ax.plot(education_heavy_series.index, education_heavy_series.values, label="Education-heavy household", color="#1baf7a", linewidth=2)
ax.plot(reference_series.index, reference_series.values, label="Illustrative comparison index", color="#eb6834", linewidth=2, linestyle="--")
ax.set_xticks(ax.get_xticks()[::6])
ax.set_ylabel("Index (base = 100)")
ax.set_title("Two hypothetical households diverge from the comparison index differently (SYNTHETIC DEMO DATA)")
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Takeaway

Even using the *same* underlying synthetic price data, two households
with different spending patterns end up with meaningfully different
personal inflation experiences -- and both can diverge from the
single-basket comparison index. This is the core mechanism this project
explores: **household heterogeneity in expenditure weights is enough,
by itself, to produce a wedge between personal and official inflation**,
even before accounting for substitution bias, regional price
differences, or basket staleness (all discussed in
`docs/METHODOLOGY.md`).

Remember: this notebook uses illustrative weights and synthetic prices
throughout. Substitute your own real expenditure shares and real price
observations (see `docs/DATA_SOURCES.md`) to get a meaningful personal
estimate.
